Step One: Process and merge Fake News Dataset and CoAID Dataset

In [91]:
import pandas as pd
import os
import glob
import re
import json

In [94]:
def is_english(text):
    text = str(text)
    if len(text.strip()) < 3:
        return False

    non_ascii_chars = len(re.findall(r'[^\x00-\x7F]', text))
    total_chars = len(text)
    if total_chars > 0 and (non_ascii_chars / total_chars) > 0.10:
        return False
        
    return True

# Process Harmful Medical Misinformation dataset
file_path = 'medmis_realistic_noisy_1600_dataset.xlsx'
df_fakenews_clean = pd.read_excel(file_path, usecols=['claim_text', 'veracity_label'])
df_fakenews_clean = df_fakenews_clean.dropna(subset=['claim_text', 'veracity_label']) # Drop rows that have missing values in these columns

# Language Filter
print(f"Size before language filter: {len(df_fakenews_clean)}")
df_fakenews_clean = df_fakenews_clean[df_fakenews_clean['claim_text'].apply(is_english)]
print(f"Size after language filter: {len(df_fakenews_clean)}")

df_fakenews_clean['veracity_label'] = df_fakenews_clean['veracity_label'].map({'True': 0, 'Fake': 1}) 
df_fakenews_clean = df_fakenews_clean.rename(columns={
    'claim_text': 'text', 
    'veracity_label': 'label'
}) 
print(df_fakenews_clean.head())

Size before language filter: 1600
Size after language filter: 1123
                                                text  label
0  Vaccines can cause mild fever, so they are uns...      0
1  Prenatal care helps monitor maternal and fetal...      0
2  Health tip: High fever in infants should be ev...      0
5  Depression can be cured by stopping all psychi...      1
6  Detox water removes all toxins from the body o...      1


In [96]:
# Process Fake News dataset: only keep columns title and label
df_fakenews2 = pd.read_csv('ACOVMD.csv', encoding='cp1252')
df_fakenews_clean2 = df_fakenews2[['Tweet', 'Label']].copy()

# Convert boolean labels True/False to 1/0, then invert
df_fakenews_clean2['Label'] = df_fakenews_clean2['Label'].astype(int)
df_fakenews_clean2['Label'] = 1 - df_fakenews_clean2['Label']

# Language Filter
print(f"Size before language filter: {len(df_fakenews_clean2)}")
df_fakenews_clean2 = df_fakenews_clean2[df_fakenews_clean2['Tweet'].apply(is_english)]
print(f"Size after language filter: {len(df_fakenews_clean2)}")

# Rename columns to match standard format
df_fakenews_clean2 = df_fakenews_clean2.rename(columns={
    'Tweet': 'text', 
    'Label': 'label'
}) 

print(df_fakenews_clean2.head())

Size before language filter: 500
Size after language filter: 500
                                                text  label
0  # COVID19 vaccines are a safer way to build pr...      0
1  Many people with disabilities have conditions ...      0
2  Counties in states with statewide mask mandate...      0
3  .@CDCMMWR finds 21 #COVID19 cases in 2 Hawaii ...      0
4  It’s important for everyone to use all the too...      0


In [97]:
# Process coAID datraset: only keep title column and create a new label column
# Use this dataset for the final test

coaid_folder_path = 'archive/ClaimFakeCOVID-19_5.csv'
coaid_files = glob.glob(coaid_folder_path) # Find all CSV files in that folder
coaid_dataframes = []

for file_path in coaid_files:
    df_temp = pd.read_csv(file_path)
    # Isolate the title and extract file name
    df_clean = df_temp[['title']].copy()
    file_name = os.path.basename(file_path)

    # Check the filename for 'Real' or 'Fake' and assign the label
    if 'Real' in file_name:
        df_clean['label'] = 0
    elif 'Fake' in file_name:
        df_clean['label'] = 1
    else:
        print(f"Warning: Could not determine label for {file_name}")
        continue # Skip if name doesn't match format
  
    df_clean = df_clean.rename(columns={'title': 'text'})
    coaid_dataframes.append(df_clean)

# Combine all individual CoAID dataframes
df_coaid = pd.concat(coaid_dataframes, ignore_index=True)
df_coaid = df_coaid.dropna(subset=['text', 'label']) # Only keep 'text' and 'label' columns
df_coaid['label'] = df_coaid['label'].astype(int) # Ensure 'label' column has int values

print(f"Size of Dataset 3: {len(df_coaid)}")
print(df_coaid.head())

Size of Dataset 3: 27
                                                text  label
0  "Spraying chlorine or alcohol on the skin kill...      1
1   "Only older adults and young people are at risk"      1
2                     "Children cannot get COVID-19"      1
3                    "COVID-19 is just like the flu"      1
4                      "Everyone with COVID-19 dies"      1


In [98]:
# Merge and Shuffle first two datasets for training, validation, and initial test
combined_df = pd.concat([df_fakenews_clean, df_fakenews_clean2], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)
combined_df['label'] = combined_df['label'].astype(int)

print(f"\nSize of combined dataset: {len(combined_df)}")
print(combined_df.head())


Size of combined dataset: 1623
                                                text  label
0  Depression can be cured by stopping all psychi...      1
1  Stop taking prescribed medicin and inhale stea...      1
2  Sugar contributes calories, so fruit should be...      0
3  #NEW: CDC has a #COVID19 toolkit available for...      0
4  A viral post says: Vaccines help the immune sy...      0


Step Two: Clean data in 'text' column by removing stopwords, special characters, and hashtags. Covert all letters to lowercase, and enforce 50 word limit

In [99]:
import re
import torch
import nltk
import sklearn
from nltk.corpus import stopwords
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.model_selection import train_test_split

# Download the standard stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sissi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [100]:
def clean_and_limit_text(text):
    # Convert to string, then lowercase
    text = str(text).lower()
    
    # Remove hashtags and URLs
    text = re.sub(r'#\w+', '', text) 
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove special characters
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Tokenize by splitting into words and remove stopwords
    words = text.split()
    cleaned_words = [word for word in words if word not in stop_words]
    
    # Enforce 50 word limit
    cleaned_words = cleaned_words[:50]
    
    return cleaned_words

combined_df['clean_tokens'] = combined_df['text'].apply(clean_and_limit_text)
df_coaid['clean_tokens'] = df_coaid['text'].apply(clean_and_limit_text)

print(combined_df.head())
print(df_coaid.head())

                                                text  label  \
0  Depression can be cured by stopping all psychi...      1   
1  Stop taking prescribed medicin and inhale stea...      1   
2  Sugar contributes calories, so fruit should be...      0   
3  #NEW: CDC has a #COVID19 toolkit available for...      0   
4  A viral post says: Vaccines help the immune sy...      0   

                                        clean_tokens  
0  [depression, cured, stopping, psychiatric, med...  
1  [stop, taking, prescribed, medicin, inhale, st...  
2  [sugar, contributes, calories, fruit, complete...  
3  [cdc, toolkit, available, k, school, administr...  
4  [viral, post, says, vaccines, help, immune, sy...  
                                                text  label  \
0  "Spraying chlorine or alcohol on the skin kill...      1   
1   "Only older adults and young people are at risk"      1   
2                     "Children cannot get COVID-19"      1   
3                    "COVID-19 is just 

Step Three: Calculate and record statistics

In [101]:
# Calculate real/fake claim counts
class_counts = combined_df['label'].value_counts()

print("Cleaned Data Statistics: ")
print(f"Total samples: {len(combined_df)}")
print(f"Real Claims (0): {class_counts.get(0, 0)}")
print(f"Fake Claims (1): {class_counts.get(1, 0)}")

print("\nExample Cleaned Sample: ")
print(f"Original: {combined_df.iloc[0]['text']}")
print(f"Cleaned & Tokenized: {combined_df.iloc[0]['clean_tokens']}")
print(f"Label: {combined_df.iloc[0]['label']}")

Cleaned Data Statistics: 
Total samples: 1623
Real Claims (0): 816
Fake Claims (1): 807

Example Cleaned Sample: 
Original: Depression can be cured by stopping all psychiatric medication!!!
Cleaned & Tokenized: ['depression', 'cured', 'stopping', 'psychiatric', 'medication']
Label: 1


Step Four: Separate data into training/validation/test (70/15/15 split) and create a vocabulary for training set

In [102]:
# Split the data
train_df, temp_df = train_test_split(combined_df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

MIN_APPEAR = 2 # Min Appearance: Words that only appear 1-2 times are treated as "Unknown"

# Count all words from training data
all_words = [word for tokens in train_df['clean_tokens'] for word in tokens]
vocab_counts = Counter(all_words)
vocab_to_int = {'<UNK>': 1} # Reserve 0 for Padding (<PAD>) and 1 for Unknown words (<UNK>)

# Map each word to an integer if it meets the minimum appearance
current_idx = 2
for word, count in vocab_counts.items():
    if count >= MIN_APPEAR:
        vocab_to_int[word] = current_idx
        current_idx += 1

def text_to_ints(tokens): # Handle unknown words
    return [vocab_to_int.get(word, 1) for word in tokens] # If word known/exists in vocab then return ID, otherwise return 1 (<UNK> ID)

# Save vocab dictionary
vocab_file = 'dataset/dataset_tensors_split/vocab_to_int.json'
with open(vocab_file, 'w') as f:
    json.dump(vocab_to_int, f)
print(f"Vocabulary saved to {vocab_file}")

# Apply conversion to all splits
train_df['numerical_tokens'] = train_df['clean_tokens'].apply(text_to_ints)
val_df['numerical_tokens'] = val_df['clean_tokens'].apply(text_to_ints)
test_df['numerical_tokens'] = test_df['clean_tokens'].apply(text_to_ints)
df_coaid['numerical_tokens'] = df_coaid['clean_tokens'].apply(text_to_ints)


Vocabulary saved to dataset/dataset_tensors_split/vocab_to_int.json


Step Five: Pad the sequences

In [103]:
SEQ_LENGTH = 50

def pad_features(numerical_tokens, seq_length):
    # Create an array of zeros and get length of token list
    features = torch.zeros(seq_length, dtype=torch.int64)
    token_len = len(numerical_tokens)
    
    # If the sentence is empty after cleaning, return zeros
    if token_len == 0:
        return features
        
    # Place the tokens into the tensor
    features[:token_len] = torch.tensor(numerical_tokens)
    return features

# Apply padding
padded_train = torch.stack(
    train_df['numerical_tokens'].apply(lambda x: pad_features(x, SEQ_LENGTH)).tolist()
)
padded_val = torch.stack(
    val_df['numerical_tokens'].apply(lambda x: pad_features(x, SEQ_LENGTH)).tolist()
)
padded_test = torch.stack(
    test_df['numerical_tokens'].apply(lambda x: pad_features(x, SEQ_LENGTH)).tolist()
)
padded_test_final = torch.stack(
    df_coaid['numerical_tokens'].apply(lambda x: pad_features(x, SEQ_LENGTH)).tolist()
)

# Extract labels as a tensor
labels_train_tensor = torch.tensor(train_df['label'].values, dtype=torch.float32)
labels_val_tensor = torch.tensor(val_df['label'].values, dtype=torch.float32)
labels_test_tensor = torch.tensor(test_df['label'].values, dtype=torch.float32)
labels_test_final_tensor = torch.tensor(df_coaid['label'].values, dtype=torch.float32)

Step Six: Wrap in Pytorch Dataset and DataLoader

In [104]:
class MedicalMisinfoDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

train_dataset = MedicalMisinfoDataset(padded_train, labels_train_tensor)
val_dataset = MedicalMisinfoDataset(padded_val, labels_val_tensor)
test_dataset = MedicalMisinfoDataset(padded_test, labels_test_tensor)
test_final_dataset = MedicalMisinfoDataset(padded_test_final, labels_test_final_tensor)

# Wrap it in a DataLoader for batching (32 samples per batch)
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_final_loader = DataLoader(test_final_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Test the loader
dataiter = iter(train_loader)
sample_x, sample_y = next(dataiter)

print("\nDataLoader Verification: ")
print(f"Batch X shape: {sample_x.shape}")
print(f"Batch Y shape: {sample_y.shape}")


DataLoader Verification: 
Batch X shape: torch.Size([32, 50])
Batch Y shape: torch.Size([32])


Step Seven: Save the cleaned and processed dataset


In [105]:
# Save to dataset_csv folder without the index column
csv_folder = 'dataset\\dataset_csv_split'
train_csv_path = os.path.join(csv_folder, "train_cleaned_medical_claims.csv")
train_df.to_csv(train_csv_path, index=False)

val_csv_path = os.path.join(csv_folder, "val_cleaned_medical_claims.csv")
val_df.to_csv(val_csv_path, index=False)

test_csv_path = os.path.join(csv_folder, "test_cleaned_medical_claims.csv")
test_df.to_csv(test_csv_path, index=False)

test_final_csv_path = os.path.join(csv_folder, "test_final_cleaned_medical_claims.csv")
df_coaid.to_csv(test_final_csv_path, index=False)

print(f"Dataframes successfully saved to: {csv_folder}")

# Save Pytorch tensors
tensor_folder = 'dataset\\dataset_tensors_split'
train_tensor_path = os.path.join(tensor_folder, "train_tensors.pt")
torch.save({
    'features': padded_train,
    'labels': labels_train_tensor
}, train_tensor_path)

val_tensor_path = os.path.join(tensor_folder, "val_tensors.pt")
torch.save({
    'features': padded_val,
    'labels': labels_val_tensor
}, val_tensor_path)

test_tensor_path = os.path.join(tensor_folder, "test_tensors.pt")
torch.save({
    'features': padded_test,
    'labels': labels_test_tensor
}, test_tensor_path)

test_final_tensor_path = os.path.join(tensor_folder, "test_final_tensors.pt")
torch.save({
    'features': padded_test_final,
    'labels': labels_test_final_tensor
}, test_final_tensor_path)

print(f"PyTorch tensors successfully saved to: {tensor_folder}")

Dataframes successfully saved to: dataset\dataset_csv_split
PyTorch tensors successfully saved to: dataset\dataset_tensors_split


In [106]:
print(f"Training tensors:   {len(train_dataset)}")
print(f"Validation tensors: {len(val_dataset)}")
print(f"Testing tensors:    {len(test_dataset)}")
print(f"Final Testing tensors:    {len(test_final_dataset)}")

#loaded_train = torch.load('dataset_tensors/train_tensors.pt')
#train_dataset = TensorDataset(loaded_train['features'], loaded_train['labels'])
#train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

Training tensors:   1136
Validation tensors: 243
Testing tensors:    244
Final Testing tensors:    27
